# Tamm-Dancoff approximation for $n$ excited state with Jaynes-Cummings ($\text{TDA}_n-\text{JC}$)

**Created by:** Khang Luong  
**Department of Chemistry, Brandeis University**  
#### **Reference:** https://doi.org/10.1063/5.0057542
---
## Overview
This tutorial serves as a continuation of the previous one (Lesson1_TDAJC) with multiple excited states models. 

By the end of this tutorial, you will be able to: 

1. **Part 1-** Understand the derivation of n-state JC models 
2. **Part 2-** Know how to compare the difference between $\text{TDA}_n-\text{JC}$ calculations
3. **Part 3-** Compare $\text{TDA}_n-\text{JC}$ and TDA-PF
4. **Part 4-** Optional Exercises
---
## Environment Setup
Please review the Lesson 1 notebook for specific requirements. After that, execute the cell below to confirm that the environment is configured correctly.

### 📦 Imports & Configuration

In [ ]:
import numpy as np
from pyscf import gto, scf, tdscf
import scipy
import qed
from qed.tdscf.ghf import FewLevel # Importing the TDA_n_JC class from the qed package
#pip install qed later. 
import matplotlib.pyplot as plt
import warnings

warnings.filterwarnings("ignore")

# Color for printing
RED    = "\033[91m"
GREEN  = "\033[92m"
YELLOW = "\033[93m"
BLUE   = "\033[94m"
RESET  = "\033[0m"

#Unit Conversion:
HF_TO_EV = 27.2114
EV_TO_HF = 1 / HF_TO_EV

print("✅ All the imports are successful!")

---
# 🗂️ Part 1: Theoretical Framework
## 1.1 Ground State calculation
You can specify the structure and basis of the molecule directly in the following code block or any structual file. Then, run the ground state SCF calculation:

In [ ]:
mol_file = 'ethene.xyz'  # Replace with your molecule file path
mol = gto.M(atom=mol_file, basis='6-311++G')
mf = scf.RKS(mol)
mf.xc = 'pbe0'
mf.kernel()

## 1.2 n-state Jaynes-Cummings (JC) models

For a two-level system made of $|e_1\rangle|0_\alpha\rangle$ and $|g\rangle|1_\alpha\rangle$ that have a resonance energy $\hbar\omega$, the two-state JC equation can be expressed as

$$
\begin{bmatrix}
\hbar\omega & s\hbar g_1\\
s\hbar g_1 & \hbar\omega
\end{bmatrix}
\begin{bmatrix}
X_1 \\ M
\end{bmatrix}
=
\hbar\Omega
\begin{bmatrix}
X_1 \\ M
\end{bmatrix},

\tag{1}
$$
where $s$ is a scaling factor to track the perturbation order. The solutions to this equation are known to be the lower and upper polaritons 

$$
|1_-\rangle = \frac{1}{\sqrt{2}}|e_1\rangle|0_\alpha\rangle-\frac{1}{\sqrt{2}}|g\rangle|1_\alpha\rangle, \\  
|1_+\rangle = \frac{1}{\sqrt{2}}|e_1\rangle|0_\alpha\rangle+\frac{1}{\sqrt{2}}|g\rangle|1_\alpha\rangle,

\tag{2}
$$
with the energies being
$$
\hbar\Omega_{1_-}=\hbar\omega-s\hbar g_1,\\
\hbar\Omega_{1_+}=\hbar\omega+s\hbar g_1

\tag{3}
$$

When introduce a third state $|e_2\rangle|0_\alpha\rangle$ with energy $\hbar\omega_2$, the corresponding JC equation is

$$
\begin{bmatrix}
\hbar\omega_2 & 0 & s\hbar g_2 \\ 
0 & \hbar\omega & s\hbar g_1 \\ 
s\hbar g_2 & s\hbar g_1 & \hbar\omega
\end{bmatrix}
\begin{bmatrix}
X_2 \\ X_1 \\ M
\end{bmatrix}
=
\hbar\Omega'
\begin{bmatrix}
X_2 \\ X_1 \\ M
\end{bmatrix}

\tag{4}
$$

From here, we recommend you review the literature under APPENDIX C: TWO-STATE AND THREE-STATE JC MODELS for the full calculation.

## 1.3 Implementation
### 1.3.1 Transition Dipole
The transition dipole moments can be obtained directly from PySCF. One can also adjust `.nroots` attribute to include more electronic state calculations.

**Note on Oscillator Strength:** The `oscillator_strength()` represents the dimensionless probability of a molecule interacting with electromagnetic radiation to undergo a specific electronic transition. A higher value indicates a brighter state. In cavity QED simulations, we typically align the cavity field polarization ($\vec{\epsilon}$) with the transition dipole moment ($\vec{d}_{ig}$) of the brightest state to achieve the maximum possible light-matter coupling strength.

Excecute the cell below to obtain the brightest excitation energy and transition dipole moments (refer to previous tutorial for more detail.

In [ ]:
td = tdscf.TDA(mf)
td.nroots = 5 # Solve for more electronic states 
td.kernel()

osc = td.oscillator_strength()
bright_idx = np.argmax(osc)
target_energy = td.e[bright_idx]
trans_dip = td.transition_dipole()[bright_idx]
print(f"Brightest excitation energy: {target_energy:.4f} a.u. ({target_energy * HF_TO_EV:.4f} eV)")
print(f"Transition dipole moment: {trans_dip} a.u.")

### 1.3.2 Setup $\text{TDA}_n-\text{JC}$
The ```FewLevel``` model is a **reduce-basis approximation** that simplifies the complex light-matter interaction by focusing only on a handful of key electronic excitations. 

To run it, we need to specify the required keys just like in TDA-JC, and in addition, explicitly specify ```cavity_model```, ```nstates```, ```target_states```, ```solver_algorithm```, ```solver_conv_prop```, ```level_shift```, ```tolerance```, and ```max_cycle```. ```nstates``` is the number of excited states that you want.

Execute the cell below.

In [ ]:
unit_dip = trans_dip / np.linalg.norm(trans_dip)

lambda_coupling = 0.05  # Coupling strength in a.u.
cavity_mode = (unit_dip * lambda_coupling).reshape(3, 1)  # Reshape to a column vector
n_electronic_states = 5

key = {
    'cavity_freq': target_energy,  # Resonant with the brightest excitation
    'cavity_mode': cavity_mode
}

# Create the TDA_JC object
cav_obj = qed.JC(mf, key)

# Initialize the FewerLevel model
qed_obj = FewLevel(td, cav_obj, key)
qed_obj.cavity_model = 'JC'
qed_obj.nstates = n_electronic_states+1
qed_obj.target_states = 'polariton'
qed_obj.solver_algorithm = 'direct'
qed_obj.solver_conv_prop = 'norm'
qed_obj.level_shift = 0.0
qed_obj.tolerance = 1e-8
qed_obj.max_cycle = 100

### 1.3.3 Run The kernel
```nstates``` in the kernel defines how many electronic excitations to include in the Hamiltonian construction.

In [ ]:
energies, vectors, trans_dips, mag_dip = qed_obj.kernel(nstates=n_electronic_states)

print(f"Number of states found: {len(energies)}")
print(f"Polaritonic energies (in eV): {energies * HF_TO_EV}")

### 1.3.4 Scanning Coupling Strength

Here we apply the same algorithm as above for many coupling strengths

In [ ]:
lambdas = np.linspace(0, 0.10, 20)

results_energies = []
photon_contribution = []

for lam in lambdas:
    print(f"{GREEN}Coupling lam = {lam:.3f} au...{RESET}", end="\r")

    # Update the cavity coupling strength
    key['cavity_mode'] = (unit_dip * lam).reshape(3, 1)
    
    # Re-initialize cavity and model
    cav_obj = qed.JC(mf, key)
    qed_obj = FewLevel(td, cav_obj, key)
    qed_obj.cavity_model = 'JC'
    qed_obj.nstates = n_electronic_states+1
    qed_obj.target_states = 'polariton'
    qed_obj.solver_algorithm = 'direct'
    qed_obj.solver_conv_prop = 'norm'
    qed_obj.level_shift = 0.0
    qed_obj.tolerance = 1e-8
    qed_obj.max_cycle = 100

    # Solve for polaritons 
    e, v, _, _ = qed_obj.kernel(nstates=n_electronic_states)

    # Extract photon weights
    wp, we = qed_obj.get_weights(v)

    results_energies.append(e)
    photon_contribution.append(wp)

    print(f"Coupling: {lam:.3f} a.u., Polariton Energies: {e}")
    print(f"{YELLOW}----------------------------------------------------------------{RESET}")

results_energies = np.array(results_energies) * HF_TO_EV  # Convert to eV
print(f"Shape of results: {results_energies.shape}") # Keep as probabilities (0 to 1)

---
# 📊 Part 2: Generate Comparison Plot

In [ ]:
import matplotlib.cm as cm

# Generate colors based on actual number of states
n_states = results_energies.shape[1]
colors = cm.tab10(np.linspace(0, 1, max(n_states, 10)))

for i in range(n_states):
    plt.plot(lambdas, results_energies[:, i], label=f'State {i+1}', color=colors[i], marker='o', markersize=4)

plt.xlabel('Coupling Strength ($\\lambda$)')
plt.ylabel('Energy (eV)')
plt.title(f'Polaritonic Energy Levels vs Coupling Strength ({n_states} states)')
plt.legend(ncol=2, fontsize='small')
plt.grid(True, alpha=0.3)

plt.show()

---
# 📋 Part 3: Tamm-Dancoff approximation Pauli-Fierz (TDA-PF) model
The key equation we are trying to solve in this model is 

$$
\begin{bmatrix}
\mathbf{A} + \Delta & \hbar \mathbf{g}^{\dagger} & \hbar \tilde{\mathbf{g}}^{\dagger} \\
\hbar \mathbf{g} & \hbar \mathbf{\omega} & 0 \\
\hbar \tilde{\mathbf{g}} & 0 & \hbar \mathbf{\omega}
\end{bmatrix}
\begin{bmatrix}
\mathbf{X} \\ \mathbf{M} \\ \mathbf{N}
\end{bmatrix}
=
\hbar\Omega^{\text{TDA-PF}}
\begin{bmatrix}
1 & 0 & 0 \\
0 & 1 & 0 \\
0 & 0 & -1 
\end{bmatrix}
\begin{bmatrix}
\mathbf{X} \\ \mathbf{M} \\ \mathbf{N}
\end{bmatrix}
$$

To implement this using the ```qed``` package, we follow the same procedure as TDA-JC, but instead using the ```.PF``` attribute

## 1. Run TDA-PF

In [ ]:
unit_dip = trans_dip / np.linalg.norm(trans_dip)

lambda_coupling = 0.1
cavity_mode = (unit_dip * lambda_coupling).reshape(3, 1)

key = {
    'cavity_mode': cavity_mode,
    'cavity_freq': np.array([target_energy])
}

cav_model = qed.PF(mf, key)
qed_td = qed.TDA(mf, td, cav_model, key)

qed_td.nroots = 5 
qed_td.kernel()
print("Polartion Energies using TDA-PF (in a.u.):", qed_td.e)

## 2. Compare TDA-JC and TDA-PF
Here, we compare the two model over a range of coupling strengths to observe the difference between them. Since PF model includes the Dipole Self-Energy (DSE) term $\frac{1}{2}(d \cdot \epsilon)^2$, it prevents the groud state energy from artificially collapsing at high coupling strength, which is a known failure in the JC model. 

In [ ]:
JC_energies = []
PF_energies = []

cavity_freqs = np.array([target_energy])

for lam in lambdas:
    print(f"{RED}Coupling lam = {lam:.3f} au...{RESET}", end="\r")
    unit_dip = trans_dip / np.linalg.norm(trans_dip)
    cavity_mode = (unit_dip * lam).reshape(3, 1)
    key = {
        'cavity_mode': cavity_mode,
        'cavity_freq': cavity_freqs
    }
    # JC model
    cav_jc = qed.JC(mf, key)
    td_jc = qed.TDA(mf, td, cav_jc, key)
    td_jc.nroots = 5
    td_jc.kernel()
    JC_energies.append(td_jc.e)

    # PF model
    cav_pf = qed.PF(mf, key)
    td_pf = qed.TDA(mf, td, cav_pf, key)
    td_pf.nroots = 5
    td_pf.kernel()
    PF_energies.append(td_pf.e)

JC_energies = np.array(JC_energies)
PF_energies = np.array(PF_energies)

plt.figure(figsize=(10, 6))
plt.plot(lambdas, JC_energies[:, 0] * HF_TO_EV, color='tab:red', marker='o', markersize=6, alpha=0.7, label='JC Lower Polariton')
plt.plot(lambdas, JC_energies[:, 1] * HF_TO_EV, color='tab:blue',marker='o', markersize=6, alpha=0.7, label='JC Upper Polariton')
plt.plot(lambdas, JC_energies[:, 2] * HF_TO_EV, color='tab:green', marker='o', markersize=6, alpha=0.7, label='JC Third Polariton')

plt.plot(lambdas, PF_energies[:, 0] * HF_TO_EV, color='tab:orange', marker='o', markersize=6, alpha=0.7, label='PF Lower Polariton')
plt.plot(lambdas, PF_energies[:, 1] * HF_TO_EV, color='tab:purple',marker='o', markersize=6, alpha=0.7, label='PF Upper Polariton')
plt.plot(lambdas, PF_energies[:, 2] * HF_TO_EV, color='tab:brown', marker='o', markersize=6, alpha=0.7, label='PF Third Polariton')

plt.axhline(y=target_energy * HF_TO_EV, color='black', linestyle=':', alpha=0.7, label='Underlying Bright State')

plt.xlabel('Coupling Strength (a.u.)')
plt.ylabel('Energy (eV)')
plt.title('Compare TDA-JC vs TDA-PF')
plt.legend(ncol=2)
plt.show()


# 🏋️ Part 4: Recommended Exercises

To deepen your understanding of the $\text{TDA}_n-\text{JC}$ model, try the following exercises:

#### **Exercise 1: Convergence with Number of States**
The `FewLevel` approximation relies on including enough electronic states to accurately capture the polariton spectrum.
*   **Task:** For a fixed coupling strength (e.g., $\lambda = 0.1$ a.u.), systematically increase the number of electronic states (`n_electronic_states`) from 1 to 10 in your `FewLevel` model.
*   **Question:** How many states does it take for the lower and upper polariton energies to converge? Does the required number of states depend on the coupling strength?

#### **Exercise 2: FewLevel vs. Full Ab Initio**
The `FewLevel` model (`solver_algorithm = 'direct'`) is an approximation of the full QED-TDA equations.
*   **Task:** Change the `solver_algorithm` from `'direct'` to `'davidson_qr'` to run the full ab initio TDA-JC calculation for $\lambda = 0.1$ a.u.
*   **Question:** How much do the polariton energies differ between the `FewLevel` approximation and the full ab initio calculation? 

#### **Exercise 3: The Rotating Wave Approximation (RWA)**
The JC model neglects both dipole self-energy and counter-rotating terms (CRTs). The RWA model includes DSE but neglects CRTs.
*   **Task:** Change the `cavity_model` parameter from `'JC'` to `'RWA'` and re-run the scan over coupling strengths.
*   **Question:** How does the inclusion of the Dipole Self-Energy in the RWA model affect the polariton energies compared to the pure JC model, especially at high coupling strengths?